# Practice 107 — Sensitivity Analysis & Partial Identification

**Theoretical context**: see `CLAUDE.md` in this folder before starting.

**Phases**: this notebook mirrors the phases in `CLAUDE.md` § Instructions.
Each phase's exercise calls into a `src/_0N_<phase_name>.py` companion module —
read that module's `TODO(human)` block before implementing it there, then
re-run the corresponding cell below.

**The pedagogical spine**: a point estimate + confidence interval only prices in
*sampling* error — it says nothing about *identification* error (whether the causal
assumption behind the estimate is even approximately right). Every phase below
computes a different way of quantifying identification error, and the final cell
puts a naive CI and the identification bounds side by side so the contrast is
impossible to miss.

## Setup

In [ ]:
import sys
from pathlib import Path

# Jupyter sets the kernel's cwd to this notebook's folder, so the practice root --
# where the `src` package lives -- is not on sys.path. Put it there.
_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

import numpy as np
import pandas as pd
import xy.pyplot as plt

from src.datasets import load_dataset, build_matched_pairs, binarize_outcome
from src.plotting import sensitivity_contour_plot, manski_bound_narrowing_plot, e_value_plot

In [ ]:
data = load_dataset(n=1000, confound_strength=1.2, seed=0)
print(f"True ATE: {data.ate_true}")
data.df.head()

## Phase 1 — Rosenbaum bounds for a matched design

We match treated and control units on the *observed* covariates `x1`, `x2` only —
never on the hidden confounder `u` in `src/datasets.py`. A matched design can look
perfectly balanced on paper and still hide a systematic imbalance in `u`. Rosenbaum's
Gamma asks: how large would that hidden imbalance have to be before the matched-pair
result stopped being significant?

In [ ]:
pair_diffs = build_matched_pairs(data)
print(f"Matched pairs: {len(pair_diffs)}, mean diff: {pair_diffs.mean():.3f}")

### Exercise — `src/_01_rosenbaum_bounds.py :: rosenbaum_pvalue_bound`

Open `src/_01_rosenbaum_bounds.py`, read the `TODO(human)` block above the function,
implement it there, then re-run the cell below.

In [ ]:
from src._01_rosenbaum_bounds import rosenbaum_pvalue_bound, find_critical_gamma

for gamma in (1.0, 1.5, 2.0, 3.0):
    p = rosenbaum_pvalue_bound(pair_diffs, gamma)
    print(f"Gamma={gamma:.1f}  upper-bound p-value={p:.4f}")

gamma_star = find_critical_gamma(pair_diffs)
print(f"\nCritical Gamma (p crosses 0.05): {gamma_star:.2f}")

### Sensitivity contour

A 2D view of the same idea: the upper-bound p-value over a grid of confounding
strength in two directions (here, Gamma vs. a synthetic second axis representing an
independent replication with resampled pair order, to make the contour meaningful as
a genuine 2D surface rather than a 1D line repeated). The red contour marks the
"explained away" frontier where the bound crosses alpha=0.05.

In [ ]:
gamma_grid = np.arange(1.0, 4.01, 0.1)
delta_grid = np.arange(0.5, 1.51, 0.05)  # relative weight on the pair-difference magnitudes
pvalue_grid = np.array([
    [rosenbaum_pvalue_bound(pair_diffs * d, g) for g in gamma_grid]
    for d in delta_grid
])
fig = sensitivity_contour_plot(gamma_grid, delta_grid, pvalue_grid)
fig

## Phase 2 — Oster's delta and the R-max bound

A complementary check using coefficient stability: how much does the treatment
coefficient move when observed controls (`x1`, `x2`) are added, relative to how much
those controls move R²? Oster's delta converts that movement into a bound on how
strong an *unobservable* confounder would need to be to explain the whole effect.

In [ ]:
from src._02_oster_delta import fit_short_and_long

oster_inputs = fit_short_and_long(data)
print(f"beta_short={oster_inputs.beta_short:.3f}  R2_short={oster_inputs.r2_short:.4f}")
print(f"beta_long={oster_inputs.beta_long:.3f}  R2_long={oster_inputs.r2_long:.4f}")

### Exercise — `src/_02_oster_delta.py :: oster_delta`

Open `src/_02_oster_delta.py`, read the `TODO(human)` block above the function,
implement it there, then re-run the cell below.

In [ ]:
from src._02_oster_delta import oster_delta

delta, beta_star = oster_delta(oster_inputs)
print(f"delta={delta:.3f}")
print(f"beta_star(delta=1)={beta_star:.3f}  (true ATE: {data.ate_true})")

## Phase 3 — E-values

Rosenbaum's Gamma and Oster's delta each answer the same question in their own
method-specific units. The E-value (Ding & VanderWeele, 2016) puts it in one common
unit — the minimum confounder-to-treatment *and* confounder-to-outcome risk-ratio
association, simultaneously, that would fully explain away the observed effect.

In [ ]:
from src._03_e_value import risk_ratio_with_ci, binarize_outcome

y1, y0 = binarize_outcome(data, threshold=65.0)
rr, ci_lo, ci_hi = risk_ratio_with_ci(y1, y0)
ci_limit = ci_lo if abs(np.log(ci_lo)) < abs(np.log(ci_hi)) else ci_hi
print(f"Risk ratio: {rr:.3f}  95% CI: ({ci_lo:.3f}, {ci_hi:.3f})")

### Exercise — `src/_03_e_value.py :: e_value`

Open `src/_03_e_value.py`, read the `TODO(human)` block above the function,
implement it there, then re-run the cell below.

In [ ]:
from src._03_e_value import e_value

e_point = e_value(rr)
e_ci = e_value(ci_limit)
print(f"E-value (point estimate): {e_point:.3f}")
print(f"E-value (CI limit closer to null): {e_ci:.3f}")

fig = e_value_plot(rr, ci_limit, e_point, e_ci)
fig

## Phase 4 — Manski worst-case bounds and Lee (2009) trimming bounds

Now we drop point identification entirely: `y_obs` is missing for some units, and
missingness depends on the outcome itself (non-ignorable attrition — see
`src/datasets.py`). No covariate adjustment fixes this. Manski's worst-case bounds
make no assumption about the missing values at all; Lee's trimming bounds add one
monotonicity assumption and are provably tighter when it holds.

In [ ]:
y_obs = data.df["y_obs"].to_numpy()
response = data.df["response"].to_numpy()
t = data.df["t"].to_numpy()
print(f"Response rate, treated: {response[t == 1].mean():.3f}  control: {response[t == 0].mean():.3f}")

### Exercise — `src/_04_manski_bounds.py :: manski_worst_case_bounds`

Open `src/_04_manski_bounds.py`, read the first `TODO(human)` block, implement it,
then re-run the cell below.

In [ ]:
from src._04_manski_bounds import manski_worst_case_bounds

manski_lo, manski_hi = manski_worst_case_bounds(y_obs, response, t, data.y_min, data.y_max)
print(f"Manski worst-case bounds: [{manski_lo:.2f}, {manski_hi:.2f}]  (true ATE: {data.ate_true})")

### Exercise — `src/_04_manski_bounds.py :: lee_trimming_bounds`

Open `src/_04_manski_bounds.py`, read the second `TODO(human)` block, implement it,
then re-run the cell below.

In [ ]:
from src._04_manski_bounds import lee_trimming_bounds

lee_lo, lee_hi = lee_trimming_bounds(y_obs, response, t)
print(f"Lee trimming bounds:      [{lee_lo:.2f}, {lee_hi:.2f}]")

## End-to-end: the naive CI vs. the identification bounds

A naive analyst who only sees `x1`, `x2`, `t`, `y_obs` would compute a point estimate
and a *sampling*-error-only confidence interval on the complete cases and stop there.
This final plot puts that naive CI next to the two identification-bound regimes from
this phase, so the gap between "looks precise" and "is actually identified" is visible
in one picture.

In [ ]:
import statsmodels.api as sm

complete = data.df.dropna(subset=["y_obs"])
naive_fit = sm.OLS(complete["y_obs"], sm.add_constant(complete[["t", "x1", "x2"]])).fit()
naive_estimate = naive_fit.params["t"]
naive_ci = tuple(naive_fit.conf_int().loc["t"])
print(f"Naive estimate: {naive_estimate:.2f}  95% CI: {naive_ci}")

fig = manski_bound_narrowing_plot(
    ["no assumption (Manski)", "monotonicity (Lee)"],
    [manski_lo, lee_lo],
    [manski_hi, lee_hi],
    naive_estimate,
    naive_ci,
)
fig

## Verification

Sanity-checks that must pass once every TODO is implemented.

In [ ]:
assert gamma_star > 1.0, "critical Gamma should exceed 1 (some robustness to hidden bias)"
assert e_point >= 1.0 and e_ci >= 1.0
assert e_ci <= e_point + 1e-9, "the CI-limit E-value should never exceed the point E-value"
assert manski_lo <= lee_lo and lee_hi <= manski_hi, "Lee bounds should sit inside Manski's"
assert manski_lo <= data.ate_true <= manski_hi, "the true ATE should fall within the worst-case bounds"
print("OK")